# Dual Access Control — resumen final del proyecto

**Objetivo:** presentar el desarrollo reproducible de un control de acceso con RFID simulado y verificación facial siamesa, desde el protocolo de datos hasta la demo web.

> Notebook liviano: no entrena, no recalibra, no abre SQLite y no usa imágenes faciales privadas. Las rutas se pueden editar en la siguiente celda.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Si el notebook se abre fuera del repositorio, reemplazar por Path(r'C:\\ruta\\dual-access-control').
ROOT = Path.cwd().resolve()
if not (ROOT / 'config' / 'model_config.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config' / 'model_config.json').exists(), 'Edita ROOT para apuntar a la raíz del repositorio.'

OUTPUTS = ROOT / 'outputs' / 'experiments'
FINAL_DIR = OUTPUTS / 'baseline_formal' / 'baseline_con_aumento'
MODEL_COMPARISON = OUTPUTS / 'model_comparison'
print(f'Raíz del proyecto: {ROOT}')

## 1. Problema y justificación de Deep Learning

Una tarjeta RFID identifica una credencial, pero no demuestra que quien la porta sea su titular. El segundo factor compara una captura facial con las referencias del usuario asociado al UID.

Las variaciones de luz, pose, escala, cámara, oclusión y apariencia son demasiado complejas para reglas manuales. Una red siamesa aprende una función de similitud entre pares y permite enrolar referencias nuevas sin reentrenar un clasificador cerrado de identidades.

## 2. Dataset y estructura

Los datos faciales son privados y no están incluidos. Esta celda solo presenta conteos agregados documentados.

In [ ]:
dataset_summary = pd.DataFrame({
    'indicador': ['Personas', 'Videos', 'Imágenes detectadas', 'Imágenes utilizables', 'Imágenes rechazadas'],
    'valor': [11, 44, 1702, 1544, 158],
})
split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'imágenes': [739, 400, 405],
    'videos': [22, 11, 11],
    'pares': [4000, 500, 500],
})
display(dataset_summary.style.hide(axis='index'))
display(split_summary.style.hide(axis='index'))

Estructura privada conceptual: `data/raw/<persona>/<vista>/<video>`, `data/processed/`, `data/pairs/` y `data/support_set/<usuario>/`. El preprocesamiento produce RGB `112×112×3`, `float32`, normalizado a `[0, 1]`. Los pares positivos pertenecen a la misma persona; los negativos, a personas diferentes.

## 3. Protocolo sin fuga

El enfoque inicial dividía pares aleatoriamente y podía repetir imágenes o frames casi idénticos entre train y evaluación. El protocolo corregido divide videos completos antes de formar pares:

`manifest → split por video → pares dentro de cada split → auditoría`

Las identidades aparecen en los tres splits, pero las sesiones de video no se cruzan. Esto mide generalización entre capturas de identidades conocidas; no equivale aún a un protocolo de identidades no vistas.

In [ ]:
leak_audit = pd.DataFrame({
    'control': ['Imágenes compartidas', 'Videos compartidos', 'Pares repetidos', 'Hashes cruzados'],
    'hallazgos': [0, 0, 0, 0],
    'estado': ['OK'] * 4,
})
display(leak_audit.style.hide(axis='index'))

## 4. Arquitectura del modelo siamés

Dos imágenes pasan por el mismo encoder —pesos compartidos—. El baseline contiene cuatro bloques `Conv2D + BatchNormalization + MaxPooling`, `Flatten`, Dense 256, Dropout 0.3 y embedding 128. La distancia L1 alimenta una salida sigmoide de similitud.

```text
imagen A ─┐                         ┌─ |embedding A - embedding B| ─ Dense sigmoid ─ score
          ├─ encoder CNN compartido ┤
imagen B ─┘                         └─ embedding B
```

El baseline tiene 4,208,257 parámetros. El modelo final no usa backbone preentrenado: se priorizaron estabilidad, comparación controlada y una demo reproducible con el dataset y tiempo disponibles.

## 5. Configuración de entrenamiento

La celda carga `training_config.json` si existe; en una copia sin outputs usa los valores documentados. No invoca TensorFlow ni entrena.

In [ ]:
training_path = FINAL_DIR / 'training_config.json'
if training_path.exists():
    training = json.loads(training_path.read_text(encoding='utf-8'))
    training_view = {
        'optimizer': 'Adam',
        'learning_rate': training.get('learning_rate'),
        'batch_size': training.get('batch_size'),
        'epochs_completed': training.get('epochs_completed'),
        'patience': training.get('early_stopping', {}).get('patience'),
        'seed': training.get('seed'),
        'augmentation_train': training.get('train_augmentation'),
        'augmentation_validation': training.get('validation_augmentation'),
    }
else:
    training_view = {'optimizer': 'Adam', 'learning_rate': 1e-4, 'batch_size': 64,
                     'epochs_completed': 10, 'patience': 5, 'seed': 42,
                     'augmentation_train': True, 'augmentation_validation': False}
display(pd.DataFrame(training_view.items(), columns=['parámetro', 'valor']).style.hide(axis='index'))

## 6. Resultados baseline

Con el threshold `max_f1` calibrado en validation, el baseline aumentado obtuvo en test limpio accuracy 0.9800, F1 0.9803, FAR 0.0360 y FRR 0.0040. Es una referencia experimental; el operating point final se ajustó después con el criterio de seguridad.

## 7. Comparación con/sin augmentation

La arquitectura y todos los hiperparámetros se mantuvieron constantes. La única diferencia fue activar aumentos realistas en train; validation y test permanecieron limpios.

In [ ]:
comparison_path = OUTPUTS / 'baseline_formal' / 'comparison' / 'clean_test_comparison.csv'
if comparison_path.exists():
    augmentation_comparison = pd.read_csv(comparison_path)
else:
    augmentation_comparison = pd.DataFrame([
        {'variant': 'con_aumento', 'accuracy': 0.980, 'f1': 0.9803, 'far': 0.036, 'frr': 0.004},
        {'variant': 'sin_aumento', 'accuracy': 0.978, 'f1': 0.9780, 'far': 0.024, 'frr': 0.020},
    ])
display(augmentation_comparison.round(4))
augmentation_comparison.set_index('variant')[[c for c in ['accuracy', 'f1', 'far', 'frr'] if c in augmentation_comparison]].plot(
    kind='bar', figsize=(8, 4), ylim=(0, 1), title='Test limpio con threshold max_f1 de cada variante'
)
plt.ylabel('métrica'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

El cambio limpio fue pequeño, pero el aumento elevó el accuracy promedio de stress de 0.8282 a 0.9502 y redujo el FRR promedio de 0.2916 a 0.0262. Por robustez, se conservó `baseline_con_aumento`.

## 8. Calibración del threshold

Los candidatos se seleccionaron exclusivamente con validation. `max_f1` optimiza desempeño agregado; `security_first` busca FAR ≤ 2% y FRR ≤ 5%, prioriza menor FAR y desempata por F1. Test no participó en la elección.

In [ ]:
threshold_path = MODEL_COMPARISON / 'threshold_metrics.csv'
if threshold_path.exists():
    thresholds = pd.read_csv(threshold_path)
    thresholds = thresholds[thresholds['model'].eq('baseline_con_aumento') & thresholds['criterion'].isin(['max_f1', 'security_first'])]
    columns = ['criterion', 'threshold', 'validation_f1', 'validation_far', 'validation_frr',
               'test_f1', 'test_far', 'test_frr']
    thresholds = thresholds[columns]
else:
    thresholds = pd.DataFrame([
        {'criterion': 'max_f1', 'threshold': 0.0187880173, 'validation_f1': 0.9841, 'validation_far': 0.024, 'validation_frr': 0.008, 'test_f1': 0.9803, 'test_far': 0.036, 'test_frr': 0.004},
        {'criterion': 'security_first', 'threshold': 0.3128704727, 'validation_f1': 0.9759, 'validation_far': 0.020, 'validation_frr': 0.028, 'test_f1': 0.9655, 'test_far': 0.020, 'test_frr': 0.048},
    ])
display(thresholds.round(4).style.hide(axis='index'))

El threshold final `0.3128704727` reduce falsos positivos de test de 9 a 5, mientras los falsos negativos suben de 1 a 12. En control de acceso se documenta este compromiso entre seguridad y fricción.

## 9. Evaluación GAP + L2 + coseno

La alternativa usa `GlobalAveragePooling2D`, embedding 128 normalizado L2 y coseno. Reduce parámetros de 4,208,257 a 1,062,400. En test limpio mejoró ligeramente, pero falló en el objetivo fotométrico.

In [ ]:
model_comparison = pd.DataFrame([
    {'modelo': 'baseline', 'parámetros': 4208257, 'accuracy_limpio': 0.966, 'f1_limpio': 0.9655,
     'far_limpio': 0.020, 'frr_limpio': 0.048, 'frr_poca_luz': 0.356, 'frr_sobreexposición': 0.320, 'frr_bajo_contraste': 0.304},
    {'modelo': 'GAP+L2+coseno', 'parámetros': 1062400, 'accuracy_limpio': 0.974, 'f1_limpio': 0.9735,
     'far_limpio': 0.008, 'frr_limpio': 0.044, 'frr_poca_luz': 1.000, 'frr_sobreexposición': 0.900, 'frr_bajo_contraste': 0.820},
])
display(model_comparison.style.hide(axis='index').format(precision=4))

La variante rechaza entre 82% y 100% de genuinos en las tres condiciones críticas. Se descartó aunque fuera más compacta y mejor en limpio. No se aplicó fine-tuning profundo; MobileNetV2 congelado y luego fine-tuning selectivo quedan como mejora futura. Knowledge distillation tampoco fue priorizada: no había un teacher validado ni una restricción de despliegue más urgente que corregir fuga, robustez e integración.

## 10. Modelo final seleccionado

**Modelo:** `baseline_formal/baseline_con_aumento`  
**Threshold:** `0.3128704727`  
**Regla facial:** `score >= threshold => MATCH`  
**Regla de acceso:** `RFID conocido y usuario activo AND rostro verificado => GRANTED`.

In [ ]:
config = json.loads((ROOT / 'config' / 'model_config.json').read_text(encoding='utf-8'))
final_metrics = pd.DataFrame([{'accuracy': 0.9660, 'precision': 0.9794, 'recall': 0.9520,
                               'f1': 0.9655, 'far': 0.0200, 'frr': 0.0480,
                               'roc_auc': 0.9838, 'tn': 245, 'fp': 5, 'fn': 12, 'tp': 238}])
print(f"{config['model_name']} | threshold={config['threshold']:.10f} | input={config['input_size']}")
display(final_metrics.style.hide(axis='index').format(precision=4))

In [ ]:
cm = [[245, 5], [12, 238]]
fig, ax = plt.subplots(figsize=(4.5, 4))
image = ax.imshow(cm, cmap='Blues')
for i, row in enumerate(cm):
    for j, value in enumerate(row):
        ax.text(j, i, value, ha='center', va='center', fontsize=13)
ax.set_xticks([0, 1], ['NO_MATCH', 'MATCH'])
ax.set_yticks([0, 1], ['NO_MATCH', 'MATCH'])
ax.set_xlabel('Predicción'); ax.set_ylabel('Etiqueta real'); ax.set_title('Matriz de confusión — test limpio')
fig.colorbar(image, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

### Curvas existentes

Se muestran solo si los PNG versionados están disponibles. No contienen rostros.

In [ ]:
for filename in ['test_clean_roc_curve.png', 'test_clean_precision_recall_curve.png']:
    path = FINAL_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=520))
    else:
        print(f'Gráfico opcional no disponible: {path.relative_to(ROOT)}')

## 11. Análisis de stress y errores

Los tests son deterministas, usan seed 2026 y alteran solo la imagen probe. No calibran el threshold ni sustituyen capturas reales.

In [ ]:
stress_path = FINAL_DIR / 'security_threshold_calibration' / 'stress_metrics_by_candidate.csv'
if stress_path.exists():
    stress = pd.read_csv(stress_path)
    stress = stress[stress['criterion'].eq('security_first')][['condition', 'accuracy', 'f1', 'far', 'frr']]
else:
    stress = pd.DataFrame([
        ('low_light', .782, .7471, .080, .356), ('overexposure', .832, .8019, .016, .320),
        ('low_contrast', .816, .7909, .064, .304), ('noise', .918, .9118, .012, .152),
        ('blur', .952, .9508, .024, .072), ('rotation_crop', .952, .9506, .020, .076),
        ('partial_occlusion', .938, .9353, .020, .104), ('synthetic_glasses', .958, .9571, .020, .064),
        ('synthetic_beard_shadow', .906, .8985, .020, .168),
    ], columns=['condition', 'accuracy', 'f1', 'far', 'frr'])
display(stress.round(4).style.hide(axis='index'))
stress.set_index('condition')[['far', 'frr']].plot(kind='bar', figsize=(10, 4), color=['#c44e52', '#4c72b0'])
plt.title('FAR y FRR por condición — security_first'); plt.ylabel('tasa'); plt.xticks(rotation=35, ha='right');
plt.tight_layout(); plt.show()

Poca luz, sobreexposición y bajo contraste elevan el FRR: usuarios legítimos pueden ser rechazados. Mitigación operativa: recaptura, luz frontal uniforme, control de calidad y referencias múltiples. Investigación futura: `validation-stress` independiente, CLAHE evaluado, más sesiones reales y MobileNetV2.

## 12. Inferencia modular

`src/inference` centraliza configuración, validación del `.keras`, preprocesamiento, comparación y decisión. La web reutiliza el mismo `model_config.json`; no mantiene un threshold paralelo.

```powershell
python -m src.inference.cli check-model --config config/model_config.json
python -m src.inference.cli verify-pair --reference referencia.jpg --capture captura.jpg
python -m src.inference.cli verify-references --capture captura.jpg --references frontal.jpg left.jpg right.jpg --strategy max
```

## 13. Demo web con RFID simulado

FastAPI + SQLite permite registrar usuarios y referencias, activar/desactivar, ingresar un UID, cargar o capturar un rostro y consultar historial.

El RFID físico se omitió por tiempo. El campo web sustituye únicamente la fuente del UID; la lógica es equivalente a la futura lectura serial: `UID → usuario → rostro → decisión → historial`. La regla dual y el registro ya están implementados.

## 14. Pruebas integrales

La suite cubre configuración, inferencia, decisión, base, rutas y flujo web. El plan manual añade cámara e iluminación difícil. Casos principales:

1. Registro correcto.
2. `GRANTED / RFID_AND_FACE_OK`.
3. `DENIED / RFID_UNKNOWN` sin ejecutar inferencia.
4. `DENIED / FACE_NO_MATCH`.
5. `DENIED / USER_INACTIVE` sin ejecutar inferencia.
6. Historial completo.
7. Cámara con carga de archivo como respaldo.
8. Iluminación difícil sin cambiar threshold.

```powershell
python -m compileall -q src tests
python -m unittest discover -s tests -v
python scripts/run_web_demo_smoke_test.py --health-only
```

## 15. Limitaciones

- 11 personas y sin validación con usuarios externos.
- Split por videos de identidades conocidas; falta protocolo de identidades no vistas.
- Stress sintético y sin `validation-stress` independiente.
- Sensibilidad a iluminación, exposición y contraste.
- Sin backbone preentrenado final, fine-tuning, distillation o liveness.
- RFID simulado; sin ESP32 físico en la demo final.
- Aplicación local sin autenticación, HTTPS ni despliegue público.

## 16. Ética y privacidad

Los rostros son datos sensibles. Se requieren consentimiento informado, propósito limitado, acceso mínimo, retención definida y eliminación segura. Fotos, videos, datasets, support sets, bases y modelos no se suben al repositorio.

El dataset pequeño puede introducir sesgos. FAR implica riesgo de acceso indebido; FRR, rechazo de una persona legítima. El prototipo es académico y necesita validación externa, análisis de equidad, seguridad contra ataques de presentación y revisión legal antes de uso real.

## 17. Conclusiones

1. Corregir la fuga fue más importante que optimizar una métrica sobre splits contaminados.
2. El augmentation aporta robustez clara aunque el test limpio cambie poco.
3. `security_first` reduce FAR con un costo medido en FRR.
4. Una arquitectura mejor en limpio puede ser peor bajo las condiciones relevantes.
5. La inferencia modular y la web demuestran el flujo dual completo sin hardware físico.

## 18. Próximos pasos

1. Capturar más personas, sesiones y condiciones reales.
2. Crear un `validation-stress` separado y reservar test.
3. Evaluar control de calidad, recaptura y CLAHE sin contaminar evaluación.
4. Comparar MobileNetV2 congelado y fine-tuning selectivo.
5. Incorporar liveness y evaluación de sesgos.
6. Conectar lector RFID/ESP32 manteniendo la misma interfaz de decisión.
7. Añadir autenticación, HTTPS, controles de acceso y validación externa antes de cualquier despliegue.